In [3]:
# Project 03 — Volatility Predictability
# Naive Volatility Benchmark

# Objective:
# Establish a minimal baseline for volatility predictability.
# Test whether a trivial persistence-based forecast provides
# any out-of-sample information under walk-forward evaluation.

# Volatility definition:
# Volatility is proxied using absolute returns.
# realized_vol_t = abs(return_t)

# Naive volatility forecast:
# The forecast for today's volatility is simply yesterday's realized volatility.
# predicted_vol_t = realized_vol_t_minus_1

# Key assumptions:
# - No model fitting
# - No parameters
# - No optimization
# - Only volatility persistence is assumed

# Evaluation principle:
# - Compare predicted_vol_t with realized_vol_t
# - Direction, return sign, accuracy, and PnL are NOT evaluated
# - This benchmark represents a strict lower bound

# Research rule:
# Any volatility model that cannot outperform this naive benchmark
# is considered empirically uninformative and rejected.


In [4]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))

In [5]:
from return_predictability.src.load_data import load_data
from return_predictability.src.returns import compute_log_returns




In [6]:
df=load_data("tsla.csv")
price=df["price"]
price.head()

Date
2010-06-29    1.59267
2010-06-30    1.58867
2010-07-01    1.46400
2010-07-02    1.28000
2010-07-06    1.07400
Name: price, dtype: float64

In [11]:
returns_t=compute_log_returns(price)
returns_t.head()

Date
2010-06-29         NaN
2010-06-30   -0.002515
2010-07-01   -0.081725
2010-07-02   -0.134312
2010-07-06   -0.175470
Name: price, dtype: float64

In [12]:
realized_volatility_t=abs(returns_t)
realized_volatility_t.head()

Date
2010-06-29         NaN
2010-06-30    0.002515
2010-07-01    0.081725
2010-07-02    0.134312
2010-07-06    0.175470
Name: price, dtype: float64

In [15]:
predicted_volatility=realized_volatility_t.shift(1)
predicted_volatility.head()

Date
2010-06-29         NaN
2010-06-30         NaN
2010-07-01    0.002515
2010-07-02    0.081725
2010-07-06    0.134312
Name: price, dtype: float64

In [17]:
error_t=realized_volatility_t - predicted_volatility
mse = (error_t ** 2).mean()
print(f"Naive Volatility Benchmark MSE: {mse:.6f}")

Naive Volatility Benchmark MSE: 0.001179


In [19]:
aligned_df=(pd.concat(
    [realized_volatility_t, predicted_volatility],axis=1,
    keys=["realized_volatility","predicted_volatility"]).dropna())
error_t=aligned_df["realized_volatility"] - aligned_df["predicted_volatility"]


In [27]:
N=len(error_t)
b=N//3
N,b

(3891, 1297)

In [28]:
e1 = error_t.iloc[:b]
e2 = error_t.iloc[b:2*b]
e3 = error_t.iloc[2*b:]


In [29]:
def block_metrics(e):
    mae=e.abs().mean()
    mse=(e**2).mean()
    return mae, mse

m1=block_metrics(e1)
m2=block_metrics(e2)
m3=block_metrics(e3)

m1, m2, m3

((np.float64(0.02211459861972895), np.float64(0.0010941031354951474)),
 (np.float64(0.02243042023966987), np.float64(0.0011657879856881205)),
 (np.float64(0.02554590030504043), np.float64(0.0012765127239248474)))

In [30]:
r1 = (e1.index.min(), e1.index.max())
r2 = (e2.index.min(), e2.index.max())
r3 = (e3.index.min(), e3.index.max())

r1, r2, r3


((Timestamp('2010-07-01 00:00:00'), Timestamp('2015-08-26 00:00:00')),
 (Timestamp('2015-08-27 00:00:00'), Timestamp('2020-10-20 00:00:00')),
 (Timestamp('2020-10-21 00:00:00'), Timestamp('2025-12-18 00:00:00')))

In [31]:
print("Day 2 — Naive benchmark stability (3 blocks)")
print(f"Block 1 {r1[0]} -> {r1[1]} | MAE={m1[0]:.6f} | MSE={m1[1]:.6f}")
print(f"Block 2 {r2[0]} -> {r2[1]} | MAE={m2[0]:.6f} | MSE={m2[1]:.6f}")
print(f"Block 3 {r3[0]} -> {r3[1]} | MAE={m3[0]:.6f} | MSE={m3[1]:.6f}")


Day 2 — Naive benchmark stability (3 blocks)
Block 1 2010-07-01 00:00:00 -> 2015-08-26 00:00:00 | MAE=0.022115 | MSE=0.001094
Block 2 2015-08-27 00:00:00 -> 2020-10-20 00:00:00 | MAE=0.022430 | MSE=0.001166
Block 3 2020-10-21 00:00:00 -> 2025-12-18 00:00:00 | MAE=0.025546 | MSE=0.001277


In [ ]:
#Naive benchmark stability (3 blocks)
#
# Purpose:
# Check whether the naive volatility benchmark (RV_{t+1} = RV_t)
# is stable across time or driven by a specific period.
#
# Interpretation:
# - MAE and MSE increase gradually from Block 1 to Block 3.
# - No sudden breakdown is observed in any sub-period.
#
# Meaning:
# - The naive benchmark behaves consistently across different
#   volatility regimes.
# - Higher errors in later blocks reflect higher market volatility,
#   not model failure.
#
# Research implication:
# This benchmark provides a stable lower bound.
# Any volatility model must outperform it across regimes,
# not only in a single period.
#
# Status:
# Naive benchmark stability check PASSED.
